# Preprocessing

## Imports

In [16]:
import random
import numpy as np
from sklearn.preprocessing import LabelEncoder
import pickle

import utils.data.train_val_test_split as split
from utils.data.queries import execute_query

import importlib

importlib.reload(split)

<module 'utils.data.train_val_test_split' from 'F:\\Software\\Dataspell\\AdvertisementGroupClassification\\utils\\data\\train_val_test_split.py'>

## Config

In [7]:
val_size = 0.2
test_size = 0.1

## Random Seed

In [3]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## Load Data

In [4]:
df = execute_query(
    "SELECT name, group_id, group_name FROM ml_dataset LIMIT :limit",
    limit=50000
)

## Separator Normalization

In [5]:
df["name_processed"] = df["name"].str.replace(r"[-_/|)(]+", " ", regex=True)

## Whitespace Normalization

In [6]:
df["name_processed"] = df["name_processed"].str.strip()
df["name_processed"] = df["name_processed"].str.replace(r"\s+", " ", regex=True)

## Train-Validation-Test Split

In [8]:
target_col = "group_id"

train_df, test_df, val_df = split.train_val_test_split(df, test_size=test_size, val_size=val_size, target_col=target_col, random_state=RANDOM_STATE)

In [17]:
splits = {
    "train": train_df,
    "val": val_df,
    "test": test_df
}

split.data_integrity_check(splits, target_col)

Dataset sizes:
train: 34,999
val  : 5,001
test : 10,000

Total: 50,000

Class coverage (number of classes per split):
train: 10 classes
All classes present
val  : 10 classes
All classes present
test : 10 classes
All classes present

Class distribution (top differences):

Top distribution shifts:
group_id
1142    0.000206
232     0.000023
405     0.000023
489     0.000023
589     0.000023
769     0.000023
973     0.000023
1277    0.000023
2173    0.000023
2663    0.000023
dtype: float64

Data integrity checks:
All checks passed




In [18]:
le = LabelEncoder()

train_df["group_id_encoded"] = le.fit_transform(train_df["group_id"])
val_df["group_id_encoded"]   = le.transform(val_df["group_id"])
test_df["group_id_encoded"]  = le.transform(test_df["group_id"])

all_encoded_classes_len = len(le.classes_)

with open("../models/artifacts/processors/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

In [19]:
out_dir = "../data/processed/"

train_df.to_csv(out_dir + "train.csv", index=False)
val_df.to_csv(out_dir + "val.csv", index=False)
test_df.to_csv(out_dir + "test.csv", index=False)